# 03 — Tutor-response evidence notebook

This notebook turns parsed results into a structured evidence base for responding to the tutor's methodological questions.

The GPT interpretation prompt now explicitly treats naïve `free_text_unparsed` outputs as expected baseline behaviour, not format failure.

In [1]:
from __future__ import annotations

import json
import os
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import HTML, Markdown, display

# Robust project-root detection: works when run from notebooks/ or project root.
CWD = Path.cwd().resolve()
PROJECT_ROOT = CWD.parent if CWD.name == "notebooks" else CWD

DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = DATA_DIR / "outputs"
PARSED_PATH = OUTPUT_DIR / "parsed_responses.jsonl"
RAW_PATH = OUTPUT_DIR / "raw_responses.jsonl"

# Optional .env loading for PyCharm/Jupyter server contexts that do not inherit shell exports.
try:
    from dotenv import load_dotenv
    load_dotenv(PROJECT_ROOT / ".env")
except ImportError:
    pass

def read_jsonl(path: Path) -> pd.DataFrame:
    records = []
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")
    with path.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            if line.strip():
                try:
                    records.append(json.loads(line))
                except json.JSONDecodeError as exc:
                    raise ValueError(f"Invalid JSON in {path} line {line_no}: {exc}") from exc
    return pd.DataFrame(records)

def pct(x):
    if x is None or pd.isna(x):
        return "NA"
    return f"{100 * float(x):.1f}%"

def safe_accuracy(df: pd.DataFrame, pred_col: str, gold_col: str) -> float | None:
    if df.empty or pred_col not in df.columns or gold_col not in df.columns:
        return None
    sub = df[df[pred_col].notna() & df[gold_col].notna()].copy()
    if sub.empty:
        return None
    return float((sub[pred_col].astype(str) == sub[gold_col].astype(str)).mean())

def display_compact_df(df: pd.DataFrame, font_size: str = "10px", max_rows: int = 50):
    """Display a compact DataFrame with smaller notebook font."""
    if df is None:
        display(Markdown("_No DataFrame to display._"))
        return
    with pd.option_context("display.max_rows", max_rows, "display.max_columns", 100, "display.max_colwidth", 100):
        display(
            df.style.set_table_styles([
                {"selector": "th", "props": [
                    ("font-size", font_size),
                    ("padding", "3px 6px"),
                    ("white-space", "nowrap"),
                ]},
                {"selector": "td", "props": [
                    ("font-size", font_size),
                    ("padding", "3px 6px"),
                    ("white-space", "nowrap"),
                ]},
            ])
        )

HTML("""
<style>
.dataframe th, .dataframe td {
    font-size: 10px !important;
    padding: 3px 6px !important;
}
.output_area pre {
    font-size: 11px !important;
}
</style>
""")

print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print("OPENAI_API_KEY loaded:", bool(os.getenv("OPENAI_API_KEY")))

PROJECT_ROOT = /Users/Shared/image_schema_llm_project
OPENAI_API_KEY loaded: True


In [2]:
# --- Optional GPT interpretation layer ---------------------------------------
# Set this to True when you want notebook cells to call the OpenAI API.
# Leave False for normal reproducible analysis without API calls.
ENABLE_GPT_INTERPRETATION = True

# Use a low-cost/current model available to your OpenAI API project.
# Change this to match the model enabled in your account, e.g. "gpt-5.4-mini".
INTERPRETATION_MODEL = os.getenv("INTERPRETATION_MODEL", "gpt-5.4-mini")

INTERPRETATION_CACHE_PATH = OUTPUT_DIR / "notebook_interpretations.json"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RESEARCH_CONTEXT = """
This is a Computational Linguistics MA project evaluating whether structured
image-schema prompting improves LLM interpretation of literal spatial,
metaphorical spatial, and weak-schema control sentences.

Important parsing convention:
- The naive prompt family is intentionally free-text and should normally appear
  as parse_status == "free_text_unparsed".
- free_text_unparsed is not a parsing failure when prompt_family == "naive".
  It is the expected baseline behaviour for ordinary paraphrase/interpretation.
- Direct-schema and structured-role prompts are the structured JSON-producing
  prompt families.
- Parse quality should be evaluated separately for naive and structured prompts.
- For schema accuracy, use records where usable_for_schema_accuracy == True.
- For literal/metaphorical/control accuracy, use records where usable_for_lm_accuracy == True.
- Partial records may still be valid for selected metrics if the relevant
  usability flag is True.
- The project does not claim that LLMs possess embodied cognition; it tests
  whether image-schema prompting provides a useful intermediate representational
  layer beyond ordinary paraphrase.
"""

def load_interpretation_cache() -> dict:
    if INTERPRETATION_CACHE_PATH.exists():
        try:
            return json.loads(INTERPRETATION_CACHE_PATH.read_text(encoding="utf-8"))
        except json.JSONDecodeError:
            return {}
    return {}

def save_interpretation_cache(cache: dict) -> None:
    INTERPRETATION_CACHE_PATH.write_text(
        json.dumps(cache, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

interpretation_cache = load_interpretation_cache()

def df_to_compact_markdown(df: pd.DataFrame, max_rows: int = 20) -> str:
    if df is None or df.empty:
        return "No rows available."
    compact = df.copy()
    for col in compact.select_dtypes(include="number").columns:
        compact[col] = compact[col].round(4)
    if len(compact) > max_rows:
        compact = compact.head(max_rows)
    return compact.to_markdown(index=False)

def _openai_interpretation_call(prompt: str, model: str = INTERPRETATION_MODEL) -> str:
    try:
        from openai import OpenAI
    except ImportError as exc:
        raise RuntimeError("Install the OpenAI SDK with: python -m pip install openai") from exc

    if not os.getenv("OPENAI_API_KEY"):
        raise RuntimeError("OPENAI_API_KEY is not set. Add it to .env or export it before starting Jupyter.")

    client = OpenAI()
    response = client.responses.create(
        model=model,
        input=prompt,
    )
    return response.output_text

def interpret_result(
    result_df: pd.DataFrame,
    *,
    title: str,
    cache_key: str,
    research_context: str = RESEARCH_CONTEXT,
    max_rows: int = 20,
    force_refresh: bool = False,
) -> str | None:
    """
    Generate or retrieve a concise GPT interpretation for a result table.

    Output requested:
    - 3 to 5 concise bullet points
    - one short summarising paragraph
    - methodological caution where relevant
    """
    if not force_refresh and cache_key in interpretation_cache:
        return interpretation_cache[cache_key]

    if not ENABLE_GPT_INTERPRETATION:
        return None

    table_md = df_to_compact_markdown(result_df, max_rows=max_rows)

    prompt = f"""
You are helping interpret results from a Computational Linguistics MA project.

Project context:
{research_context}

Result title:
{title}

Results table:
{table_md}

Critical interpretation rules:
- Do not treat naive prompt free-text outputs as parse failures.
- If parse_status == "free_text_unparsed" occurs for prompt_family == "naive",
  describe this as expected baseline behaviour.
- Evaluate JSON conformity mainly for direct_schema and structured_role_based prompts.
- Distinguish raw-data quality, parsing coverage, and semantic accuracy.
- Do not overclaim embodied cognition in LLMs.
- If a table includes both all-prompt and structured-only parse results, explain
  why structured-only results are the correct basis for JSON parsing quality.
- If partial records are present, mention whether the usability flags still
  permit them to contribute to specific metrics.

Output requirements:
- Use 3 to 5 concise bullet points.
- Then add one short summarising paragraph.
- Mention any important methodological caution.
- Prefer precise wording such as "suggests", "indicates", "is consistent with".
- Keep the whole answer under 180 words.
"""
    text = _openai_interpretation_call(prompt)
    interpretation_cache[cache_key] = text
    save_interpretation_cache(interpretation_cache)
    return text

def display_gpt_interpretation(
    result_df: pd.DataFrame,
    *,
    title: str,
    cache_key: str,
    max_rows: int = 20,
    force_refresh: bool = False,
):
    """
    Display GPT interpretation below a result cell.
    If API calls are disabled and no cached interpretation exists, display a placeholder.
    """
    text = interpret_result(
        result_df,
        title=title,
        cache_key=cache_key,
        max_rows=max_rows,
        force_refresh=force_refresh,
    )

    if text is None:
        display(Markdown(
            f"### GPT interpretation: {title}\n\n"
            "_Disabled. Set `ENABLE_GPT_INTERPRETATION = True` and rerun this cell to generate concise bullet points and a summary paragraph._"
        ))
        return None

    display(Markdown(f"### GPT interpretation: {title}\n\n{text}"))
    return text

In [3]:
parsed = read_jsonl(PARSED_PATH)

schema_df = parsed[
    (parsed["usable_for_schema_accuracy"].fillna(False) == True)
    & parsed["expected_schema_primary"].notna()
    & parsed["main_image_schema"].notna()
].copy()

lm_df = parsed[
    (parsed["usable_for_lm_accuracy"].fillna(False) == True)
    & parsed["expected_literal_or_metaphorical"].notna()
    & parsed["literal_or_metaphorical"].notna()
].copy()

print(f"Total parsed response records: {len(parsed)}")
print(f"Schema-usable records: {len(schema_df)}")
print(f"LM-usable records: {len(lm_df)}")

Total parsed response records: 5400
Schema-usable records: 3569
LM-usable records: 3600


## Evidence table: sentence type

In [4]:
schema_by_sentence = (
    schema_df.groupby("sentence_type")
    .apply(lambda g: pd.Series({
        "n_schema": len(g),
        "schema_accuracy": safe_accuracy(g, "main_image_schema", "expected_schema_primary"),
    }))
    .reset_index()
)
lm_by_sentence = (
    lm_df.groupby("sentence_type")
    .apply(lambda g: pd.Series({
        "n_lm": len(g),
        "literal_metaphorical_accuracy": safe_accuracy(g, "literal_or_metaphorical", "expected_literal_or_metaphorical"),
    }))
    .reset_index()
)
evidence_sentence_type = schema_by_sentence.merge(lm_by_sentence, on="sentence_type", how="outer")
display_compact_df(evidence_sentence_type)
sentence_type_interpretation = display_gpt_interpretation(
    evidence_sentence_type,
    title="Tutor evidence: accuracy by sentence type",
    cache_key="v2_03_tutor_accuracy_by_sentence_type",
)

,sentence_type,n_schema,schema_accuracy,n_lm,literal_metaphorical_accuracy
0,control_weak_schema,1172.000000,0.579352,1188.000000,0.309764
1,literal_spatial,1198.000000,0.808848,1206.000000,0.992537
2,metaphorical_spatial,1199.000000,0.905755,1206.000000,0.999171


### GPT interpretation: Tutor evidence: accuracy by sentence type

- Schema accuracy is highest for **metaphorical spatial** sentences (0.9058), followed by **literal spatial** sentences (0.8088), and is much lower for **weak-schema controls** (0.5794). This suggests image-schema prompts align especially well with spatial and metaphorical-spatial material.

- Literal/metaphorical classification accuracy is near ceiling for **literal spatial** (0.9925) and **metaphorical spatial** (0.9992), but much weaker for **control weak-schema** items (0.3098), consistent with controls being less naturally captured by the literal/metaphorical spatial distinction.

- The differing `n_schema` and `n_lm` values indicate that metrics were computed only where the relevant usability flags permitted inclusion; partial records can still validly contribute to one metric if usable for that metric.

- Methodologically, naive free-text outputs should not be treated as parse failures: `free_text_unparsed` is expected for the naive baseline. JSON parsing quality should be assessed mainly for the structured prompt families.

Overall, the results indicate that structured image-schema interpretation is more successful for spatial and metaphorical-spatial sentences than for weak-schema controls. This supports the usefulness of image-schema prompting as an intermediate representational layer, but it should not be taken as evidence that the model possesses embodied cognition.

## Evidence table: prompt family × sentence type

In [5]:
schema_by_prompt_sentence = (
    schema_df.groupby(["prompt_family", "sentence_type"])
    .apply(lambda g: pd.Series({
        "n_schema": len(g),
        "schema_accuracy": safe_accuracy(g, "main_image_schema", "expected_schema_primary"),
    }))
    .reset_index()
)
lm_by_prompt_sentence = (
    lm_df.groupby(["prompt_family", "sentence_type"])
    .apply(lambda g: pd.Series({
        "n_lm": len(g),
        "literal_metaphorical_accuracy": safe_accuracy(g, "literal_or_metaphorical", "expected_literal_or_metaphorical"),
    }))
    .reset_index()
)
evidence_prompt_sentence = schema_by_prompt_sentence.merge(
    lm_by_prompt_sentence,
    on=["prompt_family", "sentence_type"],
    how="outer",
)
display_compact_df(evidence_prompt_sentence)
prompt_sentence_interpretation = display_gpt_interpretation(
    evidence_prompt_sentence,
    title="Tutor evidence: prompt family by sentence type",
    cache_key="v2_03_tutor_prompt_family_by_sentence_type",
)

,prompt_family,sentence_type,n_schema,schema_accuracy,n_lm,literal_metaphorical_accuracy
0,direct_schema,control_weak_schema,578.000000,0.441176,594.000000,0.008418
1,direct_schema,literal_spatial,595.000000,0.825210,603.000000,1.000000
2,direct_schema,metaphorical_spatial,596.000000,0.912752,603.000000,0.998342
3,structured_role_based,control_weak_schema,594.000000,0.713805,594.000000,0.611111
4,structured_role_based,literal_spatial,603.000000,0.792703,603.000000,0.985075
5,structured_role_based,metaphorical_spatial,603.000000,0.898839,603.000000,1.000000


### GPT interpretation: Tutor evidence: prompt family by sentence type

- For literal and metaphorical spatial sentences, both structured prompt families perform strongly: literal/metaphorical accuracy is near ceiling, and schema accuracy is high, especially for metaphorical spatial items.

- Direct-schema prompting gives slightly higher schema accuracy than structured-role prompting for literal and metaphorical cases, but performs very poorly on control weak-schema literal/metaphorical classification.

- Structured-role prompting is more robust on control weak-schema items, with higher schema accuracy and much higher literal/metaphorical/control accuracy, suggesting better handling of cases where no strong spatial schema should be imposed.

- The differing `n_schema` and `n_lm` values indicate that usability flags matter: records can validly contribute to one metric but not the other, so schema accuracy and literal/metaphorical accuracy should be interpreted separately.

- Methodologically, JSON/parse quality should be assessed mainly for the structured prompt families; naive free-text outputs, if present elsewhere as `free_text_unparsed`, are expected baseline behaviour rather than parse failures.

Overall, the results are consistent with the claim that structured image-schema prompting can provide a useful intermediate representational layer, especially for spatial and metaphorical-spatial interpretation. However, they do not show that LLMs possess embodied cognition, and the weak-schema controls caution against overinterpreting schema assignment as semantic understanding.

## Scenario interpretation matrix

In [6]:
scenario_rows = [
    {
        "finding_pattern": "Literal and metaphorical accuracy are both high and similar",
        "interpretation": "Models can apply prompted image-schema categories across both uses in controlled short sentences.",
        "caution": "This does not prove embodied understanding; it may reflect linguistic regularities and instruction following.",
        "next_test": "Test less conventional examples and role/domain quality.",
    },
    {
        "finding_pattern": "Literal accuracy is higher than metaphorical accuracy",
        "interpretation": "Models recognise surface spatial structure more reliably than abstract conceptual mappings.",
        "caution": "Could reflect ambiguity in metaphor gold labels or insufficient prompt guidance.",
        "next_test": "Analyse source_domain and target_domain errors separately.",
    },
    {
        "finding_pattern": "Metaphorical accuracy is higher than literal accuracy",
        "interpretation": "Models may have learned image-schema terms from metaphor discourse and may over-apply them.",
        "caution": "Could be prompt-induced over-metaphorisation.",
        "next_test": "Check false positives on literal and control sentences.",
    },
    {
        "finding_pattern": "Structured role prompt improves control handling",
        "interpretation": "Role decomposition may act as an abstention constraint because the model must identify coherent schematic roles.",
        "caution": "Better control handling may trade off slightly against schema accuracy on clear cases.",
        "next_test": "Compare role completeness, control false positives, and schema-family confusion.",
    },
    {
        "finding_pattern": "Naïve prompt gives good ordinary interpretations",
        "interpretation": "Ordinary semantic paraphrase and explicit image-schema analysis are separable outcomes.",
        "caution": "The project must show what image-schema prompting adds beyond paraphrase.",
        "next_test": "Compare meaning adequacy, schema accuracy, and role/domain explicitness.",
    },
]
scenario_df = pd.DataFrame(scenario_rows)
display_compact_df(scenario_df, max_rows=10)
display_gpt_interpretation(
    scenario_df,
    title="Tutor-facing scenario interpretation matrix",
    cache_key="v2_03_tutor_scenario_matrix",
    max_rows=10,
)

,finding_pattern,interpretation,caution,next_test
0,Literal and metaphorical accuracy are both high and similar,Models can apply prompted image-schema categories across both uses in controlled short sentences.,This does not prove embodied understanding; it may reflect linguistic regularities and instruction following.,Test less conventional examples and role/domain quality.
1,Literal accuracy is higher than metaphorical accuracy,Models recognise surface spatial structure more reliably than abstract conceptual mappings.,Could reflect ambiguity in metaphor gold labels or insufficient prompt guidance.,Analyse source_domain and target_domain errors separately.
2,Metaphorical accuracy is higher than literal accuracy,Models may have learned image-schema terms from metaphor discourse and may over-apply them.,Could be prompt-induced over-metaphorisation.,Check false positives on literal and control sentences.
3,Structured role prompt improves control handling,Role decomposition may act as an abstention constraint because the model must identify coherent schematic roles.,Better control handling may trade off slightly against schema accuracy on clear cases.,"Compare role completeness, control false positives, and schema-family confusion."
4,Naïve prompt gives good ordinary interpretations,Ordinary semantic paraphrase and explicit image-schema analysis are separable outcomes.,The project must show what image-schema prompting adds beyond paraphrase.,"Compare meaning adequacy, schema accuracy, and role/domain explicitness."


### GPT interpretation: Tutor-facing scenario interpretation matrix

- The matrix supports interpreting high literal and metaphorical accuracy as evidence that prompted image-schema categories can be applied across controlled sentence types, but this only suggests useful representational scaffolding, not embodied cognition.

- If literal accuracy exceeds metaphorical accuracy, this indicates stronger handling of surface spatial structure than abstract mappings; if metaphorical accuracy is higher, it may reflect learned metaphor discourse or prompt-induced over-application, so false positives on literal/control items should be checked.

- Improved control handling under structured role prompts is consistent with role decomposition acting as an abstention constraint: requiring coherent schematic roles may reduce weak-schema false positives, though it could trade off against schema accuracy on clear cases.

- Naïve prompt outputs should not be counted as parsing failures when marked `free_text_unparsed`; this is expected baseline free-text behaviour. JSON conformity should be assessed primarily for `direct_schema` and `structured_role_based` prompts.

- Accuracy metrics should use the relevant usability flags: `usable_for_schema_accuracy == True` for schema accuracy and `usable_for_lm_accuracy == True` for literal/metaphorical/control accuracy, including partial records only where the relevant flag permits.

Overall, the results should be framed as showing whether structured image-schema prompting adds explicit schema, role, and domain information beyond ordinary paraphrase, while carefully separating raw-data quality, parsing coverage, and semantic accuracy.

'- The matrix supports interpreting high literal and metaphorical accuracy as evidence that prompted image-schema categories can be applied across controlled sentence types, but this only suggests useful representational scaffolding, not embodied cognition.\n\n- If literal accuracy exceeds metaphorical accuracy, this indicates stronger handling of surface spatial structure than abstract mappings; if metaphorical accuracy is higher, it may reflect learned metaphor discourse or prompt-induced over-application, so false positives on literal/control items should be checked.\n\n- Improved control handling under structured role prompts is consistent with role decomposition acting as an abstention constraint: requiring coherent schematic roles may reduce weak-schema false positives, though it could trade off against schema accuracy on clear cases.\n\n- Naïve prompt outputs should not be counted as parsing failures when marked `free_text_unparsed`; this is expected baseline free-text behaviour

## Draft tutor-facing summary

In [7]:
manual_summary = f"""
# Tutor-facing summary

The project is not designed to prove that LLMs possess embodied cognition. Instead, it evaluates whether image-schema prompting provides a useful intermediate representational layer beyond ordinary paraphrase.

The current analysis separates ordinary free-text interpretation from structured semantic annotation. Naïve prompt outputs are expected to appear as `free_text_unparsed` because they are preserved as free-text baselines rather than parsed as JSON. Direct-schema and structured role-based prompts are the structured-output conditions used for schema accuracy and literal/metaphorical/control classification.

The key methodological value is that image-schema prompting produces explicit and comparable fields: predicted schema, literal/metaphorical/control label, schematic roles, source/target domains, parse quality, and confidence. These can be scored against a human-validated sentence corpus.

The results should therefore be interpreted as evidence about structured semantic recoverability and prompt behaviour, not as evidence that models have human-like embodied experience. Strong performance on literal and metaphorical spatial sentences suggests that models can recover image-schema-like patterns when cues are clear. Weaker performance on weak-schema controls would indicate over-application and the need for better abstention mechanisms.
"""

cached_sentence = interpretation_cache.get("v2_03_tutor_accuracy_by_sentence_type", "")
cached_prompt = interpretation_cache.get("v2_03_tutor_prompt_family_by_sentence_type", "")

if cached_sentence or cached_prompt:
    manual_summary += "\n\n## GPT-assisted interpretation notes\n"
    if cached_sentence:
        manual_summary += "\n### Accuracy by sentence type\n" + cached_sentence + "\n"
    if cached_prompt:
        manual_summary += "\n### Prompt family by sentence type\n" + cached_prompt + "\n"

out_path = OUTPUT_DIR / "tutor_facing_summary.md"
out_path.write_text(manual_summary, encoding="utf-8")
display(Markdown(manual_summary))
print(f"Wrote: {out_path}")


# Tutor-facing summary

The project is not designed to prove that LLMs possess embodied cognition. Instead, it evaluates whether image-schema prompting provides a useful intermediate representational layer beyond ordinary paraphrase.

The current analysis separates ordinary free-text interpretation from structured semantic annotation. Naïve prompt outputs are expected to appear as `free_text_unparsed` because they are preserved as free-text baselines rather than parsed as JSON. Direct-schema and structured role-based prompts are the structured-output conditions used for schema accuracy and literal/metaphorical/control classification.

The key methodological value is that image-schema prompting produces explicit and comparable fields: predicted schema, literal/metaphorical/control label, schematic roles, source/target domains, parse quality, and confidence. These can be scored against a human-validated sentence corpus.

The results should therefore be interpreted as evidence about structured semantic recoverability and prompt behaviour, not as evidence that models have human-like embodied experience. Strong performance on literal and metaphorical spatial sentences suggests that models can recover image-schema-like patterns when cues are clear. Weaker performance on weak-schema controls would indicate over-application and the need for better abstention mechanisms.


## GPT-assisted interpretation notes

### Accuracy by sentence type
- Schema accuracy is highest for **metaphorical spatial** sentences (0.9058), followed by **literal spatial** sentences (0.8088), and is much lower for **weak-schema controls** (0.5794). This suggests image-schema prompts align especially well with spatial and metaphorical-spatial material.

- Literal/metaphorical classification accuracy is near ceiling for **literal spatial** (0.9925) and **metaphorical spatial** (0.9992), but much weaker for **control weak-schema** items (0.3098), consistent with controls being less naturally captured by the literal/metaphorical spatial distinction.

- The differing `n_schema` and `n_lm` values indicate that metrics were computed only where the relevant usability flags permitted inclusion; partial records can still validly contribute to one metric if usable for that metric.

- Methodologically, naive free-text outputs should not be treated as parse failures: `free_text_unparsed` is expected for the naive baseline. JSON parsing quality should be assessed mainly for the structured prompt families.

Overall, the results indicate that structured image-schema interpretation is more successful for spatial and metaphorical-spatial sentences than for weak-schema controls. This supports the usefulness of image-schema prompting as an intermediate representational layer, but it should not be taken as evidence that the model possesses embodied cognition.

### Prompt family by sentence type
- For literal and metaphorical spatial sentences, both structured prompt families perform strongly: literal/metaphorical accuracy is near ceiling, and schema accuracy is high, especially for metaphorical spatial items.

- Direct-schema prompting gives slightly higher schema accuracy than structured-role prompting for literal and metaphorical cases, but performs very poorly on control weak-schema literal/metaphorical classification.

- Structured-role prompting is more robust on control weak-schema items, with higher schema accuracy and much higher literal/metaphorical/control accuracy, suggesting better handling of cases where no strong spatial schema should be imposed.

- The differing `n_schema` and `n_lm` values indicate that usability flags matter: records can validly contribute to one metric but not the other, so schema accuracy and literal/metaphorical accuracy should be interpreted separately.

- Methodologically, JSON/parse quality should be assessed mainly for the structured prompt families; naive free-text outputs, if present elsewhere as `free_text_unparsed`, are expected baseline behaviour rather than parse failures.

Overall, the results are consistent with the claim that structured image-schema prompting can provide a useful intermediate representational layer, especially for spatial and metaphorical-spatial interpretation. However, they do not show that LLMs possess embodied cognition, and the weak-schema controls caution against overinterpreting schema assignment as semantic understanding.


Wrote: /Users/Shared/image_schema_llm_project/data/outputs/tutor_facing_summary.md
